In [7]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import TimestampedGeoJson
from shapely.geometry import Point
from pathlib import Path
from typing import Optional

# =========================
# INPUT
# =========================
results_root = Path("/Volumes/ChaikaT7/LT_CCD_damage_assessment/Regions/Lebanon/Results/Lebanon_South/SLC_1")
output_map_path = results_root / "Lebanon_South_damage_timeslider_circles.html"

# Folder for exported circle geometries
circle_output_dir = results_root / "circle_geometries"

# Show only reviewed destruction clusters
only_destruction = False

# Use building footprints filter
use_building_footprints = True

# Building footprints GeoJSON
building_footprints_path = Path("/Volumes/ChaikaT7/LT_CCD_damage_assessment/Regions/Lebanon/Data/Row_Data/Lebanon_Buildings/Lebanon_Buildings.geojson")

# Initial zoom
zoom_start = 12

# Fallback radius in meters for invalid / missing area
fallback_radius_m = 50

# Circle smoothness: more points = smoother circles, but larger HTML
circle_resolution = 128

# Visual enlargement factor for circles
scale_factor = 2

# Minimum number of building footprints that a circle must intersect to be kept
min_intersecting_buildings = 4


# =========================
# HELPERS
# =========================
def extract_post_date_from_folder(folder_name: str) -> Optional[str]:
    """
    Extract date from folder names like:
    LTCCD_Results_post15032026_conf27032026
    -> 2026-03-15
    """
    m = re.search(r"post(\d{2})(\d{2})(\d{4})", folder_name)
    if not m:
        return None
    dd, mm, yyyy = m.groups()
    return f"{yyyy}-{mm}-{dd}"


def get_radius_m(row) -> float:
    """
    Convert polygon area to equivalent-circle radius in meters:
    area = pi * r^2  ->  r = sqrt(area / pi)
    """
    area = row.get("area_m2", None)

    if area is None or pd.isna(area) or area <= 0:
        return fallback_radius_m

    return float(np.sqrt(area / np.pi)) * scale_factor


def make_circle_polygon_wgs84(lon: float, lat: float, radius_m: float, resolution: int = 32):
    """
    Build a true circle polygon around a lon/lat point using metric buffering in EPSG:3857,
    then transform it back to WGS84 (EPSG:4326).
    """
    point_wgs84 = gpd.GeoSeries([Point(lon, lat)], crs=4326)
    point_3857 = point_wgs84.to_crs(3857)

    circle_3857 = point_3857.buffer(radius_m, resolution=resolution)
    circle_wgs84 = circle_3857.to_crs(4326)

    return circle_wgs84.iloc[0]


# =========================
# PREPARE OUTPUT FOLDER
# =========================
circle_output_dir.mkdir(parents=True, exist_ok=True)
print(f"📁 Circle geometry output folder: {circle_output_dir}")


# =========================
# LOAD BUILDING FOOTPRINTS
# =========================
footprints_gdf = None

if use_building_footprints:
    if not building_footprints_path.exists():
        raise FileNotFoundError(f"Building footprints file not found: {building_footprints_path}")

    footprints_gdf = gpd.read_file(building_footprints_path)

    if footprints_gdf.empty:
        raise ValueError("Building footprints file is empty.")

    footprints_gdf = footprints_gdf[
        footprints_gdf.geometry.notnull() & ~footprints_gdf.geometry.is_empty
    ].copy()

    if footprints_gdf.empty:
        raise ValueError("No valid building footprint geometries found.")

    if footprints_gdf.crs is None:
        print("⚠️ Building footprints CRS is missing. Assuming EPSG:4326.")
        footprints_gdf = footprints_gdf.set_crs(4326)

    footprints_gdf = footprints_gdf.to_crs(4326)

    # Fix invalid geometries
    footprints_gdf["geometry"] = footprints_gdf.geometry.buffer(0)

    print(f"🏠 Loaded building footprints: {len(footprints_gdf):,}")


# =========================
# FIND ALL REVIEW FILES
# =========================
review_files = sorted(results_root.rglob("cluster_filtering/ltccd_damage_polygons_with_cluster_id.geojson"))

if len(review_files) == 0:
    raise FileNotFoundError(
        f"No ltccd_damage_polygons_with_cluster_id.geojson files found under: {results_root}"
    )

print(f"✅ Found {len(review_files)} review files")


# =========================
# LOAD AND BUILD FEATURES
# =========================
all_features = []
all_points = []
used_dates = []

for review_file in review_files:
    result_folder = review_file.parent.parent
    result_name = result_folder.name
    date_str = extract_post_date_from_folder(result_name)

    if date_str is None:
        print(f"⚠️ Skipping {review_file} (could not parse date from folder name)")
        continue

    gdf = gpd.read_file(review_file)

    if gdf.empty:
        print(f"⚠️ Skipping empty file: {review_file}")
        continue

    # Clean
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    if gdf.empty:
        print(f"⚠️ Skipping file with no valid geometries: {review_file}")
        continue

    if gdf.crs is None:
        print(f"⚠️ CRS missing in {review_file}. Assuming EPSG:4326.")
        gdf = gdf.set_crs(4326)

    gdf = gdf.to_crs(4326)

    # Fix datetime columns for JSON/GeoJSON export
    for col in gdf.columns:
        if str(gdf[col].dtype).startswith("datetime"):
            gdf[col] = gdf[col].astype(str)

    gdf = gdf.replace({np.nan: None})

    if only_destruction and "review_type" in gdf.columns:
        gdf = gdf[gdf["review_type"] == "destruction"].copy()

    if len(gdf) == 0:
        print(f"ℹ️ No matching features in {review_file}")
        continue

    # Compute centroids safely in projected CRS, then convert to WGS84
    gdf_proj = gdf.to_crs(3857).copy()
    gdf_proj["centroid"] = gdf_proj.geometry.centroid
    centroids_wgs84 = gpd.GeoSeries(gdf_proj["centroid"], crs=3857).to_crs(4326)

    gdf["centroid_lon"] = centroids_wgs84.x
    gdf["centroid_lat"] = centroids_wgs84.y

    added_count = 0
    skipped_no_footprint_count = 0
    circle_rows = []

    for _, row in gdf.iterrows():
        lat = row["centroid_lat"]
        lon = row["centroid_lon"]
        radius_m = get_radius_m(row)

        cluster_id = row.get("cluster_id_seq", row.get("cluster_id", "n/a"))
        review_type = row.get("review_type", "n/a")
        area_m2 = row.get("area_m2", None)
        delta_val = row.get("delta_gamma_mean", None)
        z_val = row.get("z_score_mean", None)

        # Build true circle polygon in meters
        circle_geom = make_circle_polygon_wgs84(
            lon=lon,
            lat=lat,
            radius_m=radius_m,
            resolution=circle_resolution
        )

        # Keep only circles that intersect at least the required number of building footprints
        intersecting_buildings_count = None

        if use_building_footprints:
            possible_matches_idx = list(
                footprints_gdf.sindex.query(circle_geom, predicate="intersects")
            )

            if len(possible_matches_idx) == 0:
                intersecting_buildings_count = 0
            else:
                possible_matches = footprints_gdf.iloc[possible_matches_idx]

                # Spatial index gives candidates; this checks real intersections
                intersecting_buildings_count = int(
                    possible_matches.intersects(circle_geom).sum()
                )

            if intersecting_buildings_count < min_intersecting_buildings:
                skipped_no_footprint_count += 1
                continue

        popup_html = (
            f"<b>Date:</b> {date_str}<br>"
            f"<b>Cluster:</b> {cluster_id}<br>"
            f"<b>Type:</b> {review_type}<br>"
            f"<b>Area (m²):</b> {area_m2 if area_m2 is not None else 'n/a'}<br>"
            f"<b>Equivalent radius (m):</b> {round(radius_m, 2)}<br>"
            f"<b>Intersecting buildings:</b> {intersecting_buildings_count if intersecting_buildings_count is not None else 'n/a'}<br>"
            f"<b>Delta:</b> {round(delta_val, 3) if delta_val is not None else 'n/a'}<br>"
            f"<b>Z-score:</b> {round(z_val, 3) if z_val is not None else 'n/a'}"
        )

        # For map
        feature = {
            "type": "Feature",
            "geometry": circle_geom.__geo_interface__,
            "properties": {
                "time": date_str,
                "popup": popup_html,
                "style": {
                    "fillColor": "red",
                    "fillOpacity": 0.20,
                    "color": "red",
                    "opacity": 0.7,
                    "weight": 1.5
                }
            }
        }

        all_features.append(feature)
        all_points.append((lat, lon))
        added_count += 1

        # For exported GeoJSON
        circle_rows.append({
            "date": date_str,
            "source_file": str(review_file),
            "result_folder": result_name,
            "cluster_id": cluster_id,
            "review_type": review_type,
            "area_m2": area_m2,
            "radius_m": radius_m,
            "delta_gamma_mean": delta_val,
            "z_score_mean": z_val,
            "centroid_lon": lon,
            "centroid_lat": lat,
            "geometry": circle_geom,
            "intersecting_buildings": intersecting_buildings_count
        })

    # Save one GeoJSON with circle geometries for this date
    if circle_rows:
        circles_gdf = gpd.GeoDataFrame(circle_rows, geometry="geometry", crs="EPSG:4326")
        circle_geojson_path = circle_output_dir / f"damage_circles_{date_str}.geojson"
        circles_gdf.to_file(circle_geojson_path, driver="GeoJSON")
        print(f"   💾 Saved circle geometries: {circle_geojson_path.name} ({len(circles_gdf):,} features)")

    if added_count > 0:
        used_dates.append(date_str)

    print(f"   Added {added_count:,} clusters from {result_name} ({date_str})")

    if use_building_footprints:
        print(f"   Skipped {skipped_no_footprint_count:,} circles without building footprint intersection")


if len(all_features) == 0:
    raise ValueError("No features available to build the time-slider map.")


# =========================
# MAP CENTER
# =========================
center_lat = float(np.mean([p[0] for p in all_points]))
center_lon = float(np.mean([p[1] for p in all_points]))

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=zoom_start,
    tiles="cartodbpositron"
)


# =========================
# TIME SLIDER LAYER
# =========================
TimestampedGeoJson(
    {
        "type": "FeatureCollection",
        "features": all_features
    },
    period="P1D",
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=1,
    loop_button=True,
    date_options="YYYY-MM-DD",
    time_slider_drag_update=True,
    duration=None
).add_to(m)


# =========================
# SAVE
# =========================
m.save(output_map_path)

print(f"✅ Time-slider map saved: {output_map_path}")
print(f"📅 Dates included: {sorted(set(used_dates))}")
print(f"📁 Circle GeoJSONs saved in: {circle_output_dir}")

📁 Circle geometry output folder: /Volumes/ChaikaT7/LT_CCD_damage_assessment/Regions/Lebanon/Results/Lebanon_South/SLC_1/circle_geometries
🏠 Loaded building footprints: 787,735
✅ Found 5 review files
   💾 Saved circle geometries: damage_circles_2026-03-02.geojson (1 features)
   Added 1 clusters from LTCCD_Results_post02032026 (2026-03-02)
   Skipped 0 circles without building footprint intersection
   💾 Saved circle geometries: damage_circles_2026-04-07.geojson (143 features)
   Added 143 clusters from LTCCD_Results_post07042026 (2026-04-07)
   Skipped 292 circles without building footprint intersection
   💾 Saved circle geometries: damage_circles_2026-04-13.geojson (170 features)
   Added 170 clusters from LTCCD_Results_post13042026 (2026-04-13)
   Skipped 266 circles without building footprint intersection
   💾 Saved circle geometries: damage_circles_2026-03-14.geojson (18 features)
   Added 18 clusters from LTCCD_Results_post14032026 (2026-03-14)
   Skipped 47 circles without buildi